In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
from torch.utils.data import DataLoader

from src.dataset import DownscaleDataset
from src.model_cnn import UNetDownscale

ImportError: cannot import name 'UNetDownscale' from 'src.model_cnn' (/Users/xichen/Documents/Courses/2026_Spring/deep_learning/hw/coding/final_project/CSCI1470-final-project/src/model_cnn.py)

In [3]:
train_path = "../.data/downscaling_splits/train_norm.nc"
val_path = "../.data/downscaling_splits/val_norm.nc"
test_path = "../.data/downscaling_splits/test_norm.nc"
topo_path = "../.data/ETOPO2/topography_features_on_gridmet_masked_norm.nc"

In [4]:
train_ds = DownscaleDataset(train_path, topo_path)

x, y, mask = train_ds[0]

print("x shape   :", x.shape)
print("y shape   :", y.shape)
print("mask shape:", mask.shape)
print("dtype     :", x.dtype, y.dtype, mask.dtype)

x shape   : torch.Size([2, 240, 311])
y shape   : torch.Size([1, 240, 311])
mask shape: torch.Size([1, 240, 311])
dtype     : torch.float32 torch.float32 torch.float32


In [5]:
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

xb, yb, mb = next(iter(train_loader))
print("xb:", xb.shape)
print("yb:", yb.shape)
print("mb:", mb.shape)

xb: torch.Size([4, 2, 240, 311])
yb: torch.Size([4, 1, 240, 311])
mb: torch.Size([4, 1, 240, 311])


In [6]:
print("xb has nan:", torch.isnan(xb).any().item())
print("yb has nan:", torch.isnan(yb).any().item())
print("mb has nan:", torch.isnan(mb).any().item())

print("xb has inf:", torch.isinf(xb).any().item())
print("yb has inf:", torch.isinf(yb).any().item())
print("mb has inf:", torch.isinf(mb).any().item())

xb has nan: False
yb has nan: False
mb has nan: False
xb has inf: False
yb has inf: False
mb has inf: False


In [7]:
model = SimpleCNN(in_channels=2)

out = model(x.unsqueeze(0))

print("input shape :", x.unsqueeze(0).shape)
print("output shape:", out.shape)
print("target shape:", y.unsqueeze(0).shape)

input shape : torch.Size([1, 2, 240, 311])
output shape: torch.Size([1, 1, 240, 311])
target shape: torch.Size([1, 1, 240, 311])


In [8]:
model = SimpleCNN(in_channels=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

xb, yb, mb = next(iter(train_loader))
pred = model(xb)

loss = ((pred - yb) ** 2 * mb).sum() / mb.sum().clamp_min(1.0)
print("loss:", loss.item())

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("One training step passed.")

loss: 0.9132742285728455
One training step passed.


In [9]:
print("pred has nan:", torch.isnan(pred).any().item())
print("pred has inf:", torch.isinf(pred).any().item())

pred has nan: False
pred has inf: False
